# Energy Management in a Microgrid via Moreau Envelope Learning ADMM.

Implementation of the Energy Management in a Microgrid example from the paper:

[LEAF: A Learning-Enabled ADMM Framework for Accelerated
Convex Optimization](https://)

**LEAF** accelerates ADMM by replacing the expensive proximal update with a learned Moreau-envelope model. For a convex objective $f$, the Moreau envelope is

$$
M_\lambda f(q)=\min_z\left(f(z)+\frac{1}{2\lambda}\|q-z\|_2^2\right),
$$

and its gradient recovers the proximal update through

$$
\operatorname{prox}_{\lambda f}(q)=q-\lambda\nabla M_\lambda f(q).
$$

Instead of learning a full solution map, LEAF learns the scalar function $M_{1/\rho}f$ with an input-convex neural network (ICNN). The learned gradient is then embedded into ADMM, producing MEL-ADMM and its split variant sMEL-ADMM.

# Problem formulation
We consider a localized energy system equipped with a battery energy storage system (BESS), a renewable generator, and a mix of controllable and uncontrollable loads.
The microgrid energy-management problem schedules battery charging/discharging and controllable load consumption over a receding model predictive control (MPC) horizon. At prediction step $k$ from current time $t$, the main decision variables are imported grid power $m_{k|t}$, BESS power $u_{k|t}$, controllable load power $p^c_{k|t}$, and BESS state of charge $x_{k|t}$. Positive $u_{k|t}$ denotes discharging, while negative $u_{k|t}$ denotes charging.

The explicit MPC optimization problem is
$$
\begin{aligned}
\min_{\{m_{k|t},u_{k|t},p^c_{k|t},x_{k|t}\}} \quad
& \sum_{k=1}^{N}
\Bigg[
r_{ec}\Delta_T\left(m_{k|t} + \frac{1-\eta}{2\sqrt{\eta}}|u_{k|t}|\right)
+ r_{op}\max(m_{k|t},0)
+ r_{df}\max\left(\frac{a}{p^c_{k|t}}-1,0\right)
\Bigg] \\
\text{s.t.} \quad
& x_{k+1|t} = x_{k|t} - \frac{\Delta_T}{b_{ess}}u_{k|t}, \qquad k=0,\ldots,N-1, \\
& p^c_{k|t} + \hat{p^u}_{k|t} = m_{k|t} + u_{k|t} + \hat g_{k|t}, \qquad k=1,\ldots,N, \\
& u_{min} \le u_{k|t} \le u_{max}, \qquad k=1,\ldots,N, \\
& x_{min} \le x_{k|t} \le x_{max}, \qquad k=0,\ldots,N, \\
& p^c_{k|t} > 0, \qquad k=1,\ldots,N, \\
& x_{0|t}=x_t, \qquad x_{N|t} \ge x_N.
\end{aligned}
$$
The quantities $\hat g_{k|t}$ and $\hat{p^u}_{k|t}$ are the forecasted renewable generation and uncontrollable demand. The parameters $r_{ec}$, $r_{op}$, and $r_{df}$ weight energy cost, peak-demand cost, and discomfort cost; $\Delta_T$ is the sampling interval; $\eta$ is BESS round-trip efficiency.


# Generating training data
**Julia kernel**

We first generate supervised data for the Moreau Envelope Learning (MEL) model. ADMM is run on a few initial battery SOC values, and each iteration provides samples of $q^i,\quad M_{1/\rho}f(q^i),\quad \nabla M_{1/\rho}f(q^i).$These are stored as `input`, `env`, and `grad`.

*This section intentionally generate only a small number of initial SOC cases and save as a demo data file. Our purpose is to help users can understand the full data-generation workflow without waiting for a large offline data-collection run.
If you want to generate a complete and full dataset, please refer to and run the generate_em.jl file located in the project folder.*

We begin by activating the Julia project



In [1]:
## Enable this cell when using Google Colab
# ;git clone https://github.com/trinhtran1120/L2O_tutorial.git

In [ ]:
## Use this path when using Google Colab
# path = "/content/L2O_tutorial/LEAF"

## Use this path when using IDE
# path = "LEAF"

"LEAF"

In [ ]:
# Import Julia packages
import Pkg
Pkg.activate("/content/L2O_tutorial/LEAF")
Pkg.instantiate()

cd("/content/L2O_tutorial/LEAF")
using JSON3, LinearAlgebra, NPZ, Printf, SparseArrays, StaticArrays, NNlib
using Plots, StatsPlots, Statistics
using Base.Threads

import MathOptInterface as MOI
import ParametricOptInterface as POI

  Activating project at `~/Library/CloudStorage/OneDrive-UniversityofCentralFlorida/09. Github/L2O_tutorial/LEAF`


and loading the local code. These files define the problem data, classical solvers, ADMM alogrithm, and data collection utilities.

In [16]:
include.(["setup_em.jl", "preprocess.jl", "solver_em.jl", "admm_em.jl"])

4-element Vector{Function}:
 energy_mag (generic function with 1 method)
 pick_solver (generic function with 3 methods)
 get_benchmark (generic function with 2 methods)
 aux_solver_eco_data (generic function with 1 method)

Next, set the ADMM parameters for data generation.

In [ ]:
# ADMM settings for generating a dataset.
max_iter = 1000     # maximum ADMM iterations
tol = 1e-3          # ADMM stopping tolerance

# Store the generated dataset under the local datasets/ folder.
const DATASET_DIR = "datasets"

Build the default microgrid data object and initialize the dictionary that will store Moreau-envelope training samples.

In [ ]:
# Build the energy-management MPC data object.
mpc_data = energy_mag()

# Appends one proximal sample at a time to this dictionary.
data_train = Dict("input" => Vector{Float64}[], "env" => Float64[], "grad" => Vector{Float64}[])

Activate Gurobi license for solving the optimization problem.

In [ ]:
ENV["GRB_LICENSE_FILE"] = "/content/L2O_tutorial/LEAFgurobi.lic"

using Gurobi

With the problem data ready, we construct the reference solver and the ADMM solvers used to collect proximal samples.

In [ ]:
# Reference MPC solver used to compute the optimal objective.
mpc_eco_sol = mpc_eco_solver("Ipopt", mpc_data, 1e-6)

# Data-generation ADMM solver.
prime_solver = prime_solver_eco_data("Ipopt", mpc_data)
aux_solver = aux_solver_eco_data("Gurobi", mpc_data)
admm_sol = ADMM_eco_iter_data(mpc_data, prime_solver, aux_solver; max_iter = max_iter, tol = tol)

Select the forecast window and the initial SOC values used for training data generation. Each SOC creates one problem instance.


In [ ]:
# Use the first MPC horizon of forecasted uncontrollable load and renewable generation.
load = mpc_data.load_forecast[1:mpc_data.N]
gen = mpc_data.gen_forecast[1:mpc_data.N]

# Set of initial battery states of charge.
train_pool = [1/2, 2/3]

Run ADMM for each selected SOC and collect the proximal samples generated along the ADMM trajectory. The printed optimality gap checks that the data-generating ADMM run is feasible.

In [ ]:
# Run ADMM for each initial SOC and collect proximal samples from every ADMM iteration.
for x0 in train_pool
    println("Collecting training data with initial SOC = $(round(100x0, digits=1))%")
    _, _, J_opt = mpc_eco_sol(x0, load, gen)
    _, J_ADMM = admm_sol(data_train, x0; verbose = true)
    @printf("opt_gap = %5.3f%%\n", abs(J_opt - J_ADMM) / abs(J_opt) * 100)
end

After collecting samples, convert them into arrays and save the MEL training dataset as `.npz`. This file is the bridge from Julia data generation to Python/JAX model training.

In [ ]:
# Convert the collected vectors into arrays for training model.
data = Dict(
    "input" => reduce(hcat, data_train["input"]),
    "grad" => reduce(hcat, data_train["grad"]),
    "rho" => mpc_data.rho,
    "env" => data_train["env"],
)

# Save the dataset.
output_path = joinpath(DATASET_DIR, "eco_mpc-demo-train.npz")
npzwrite(output_path, data)

@printf("Collected %d training data points\n", length(data_train["input"]))
println("Saved demo dataset to $(output_path)")

# Train Moreau Envelope Learning Model
**Python kernel**

The Moreau Envelope Learning (MEL) model approximates the scalar Moreau envelope of the stage cost. Following the LEAF paper, we use an [input-convex neural network (ICNN)](https://arxiv.org/abs/1609.07152) so the learned function remains convex with respect to its input. After training, ADMM can recover an approximate proximal update from the gradient of the learned envelope, replacing an optimization solve with a fast neural-network gradient evaluation.

We train the ICNN on the samples generated above: `input` contains the envelope query $q^i$, `enve` contains the Moreau-envelope value $M_{\frac{1}{\rho}} f(q^i)$, and `grad` contains the gradient target $\nabla M_{\frac{1}{\rho}} f(q^i)$. For an ICNN approximation $\hat M_{\frac{1}{\rho}}$, the training loss is

$$
L(\theta) =\sum_{i=1}^{n_D} \left(\hat M_{\frac{1}{\rho}}f(q^i) - M_{\frac{1}{\rho}} f(q^i)\right)^2 + \sum_{i=1}^{n_D}\left( \mu_g \|\nabla \hat M_{\frac{1}{\rho}}f(q^i) - \nabla M_{\frac{1}{\rho}} f(q^i)\|_2^2  + \mu_p\| \text{max}(0, \hat M_{\frac{1}{\rho}}f(q^i) - f(q^i))\|_2^2 \right),
$$

where $\mu_g > 0$ and $\mu_p > 0$.

*The training cells below use only a small subset of the demo samples generated in the previous section and train a lightweight demo ICNN. This is meant to show the execution pipeline: load MEL data, train the ICNN, monitor value/gradient loss, and export a model.*

Switch to the Python/JAX training environment and import the packages needed for ICNN training and model export.

In [ ]:
# Import Python packages
from pathlib import Path
import json
import jax
import jax.numpy as jnp
import numpy as np
import optax

Load the proximal samples generated by ADMM. `X` stores the query points $q$, `y` stores envelope values, `g` stores envelope gradients, and `rho` stores the ADMM penalty.

In [ ]:
# Load the proximal samples generated in the previous section.
train_path = Path("/content/L2O_tutorial/LEAF/datasets") / "eco_mpc-demo-train.npz"

with np.load(train_path) as data:
    X = jnp.asarray(data["input"].T[:64], dtype=jnp.float32)
    y = jnp.asarray(data["env"][:64], dtype=jnp.float32)
    g = jnp.asarray(data["grad"].T[:64], dtype=jnp.float32)
    rho = float(data["rho"])

Set the ICNN training hyperparameters.

In [ ]:
# ICNN settings.
widths = [16, 16]
learning_rate = 1e-3
grad_weight = 5.0
l2_reg = 0.0
batch_size = 16
epochs = 500
seed = 0

*If you want to train the model with a complete and full dataset, please replace `eco_mpc-demo-train.npz` with your own fully generated data file. Additionally, make sure to adjust the network hyperparameters.*

Initialize ICNN parameters.  Nonnegative transformed weights are used later in the forward pass so the learned scalar function remains convex in its input.

In [ ]:
# Initialize the ICNN parameters.
def init_icnn(key, n_in=3, widths=[16, 16]):
    # Split one random key so each parameter group is initialized independently.
    keys = jax.random.split(key, 4)

    return {
        "U1": 0.1 * jax.random.normal(keys[0], (widths[0], n_in)),
        "U2": 0.1 * jax.random.normal(keys[1], (widths[1], n_in)),
        "W2": 0.1 * jax.random.normal(keys[2], (widths[1], widths[0])),
        "b1": jnp.zeros(widths[0]),
        "b2": jnp.zeros(widths[1]),
        "v": 0.1 * jax.random.normal(keys[3], (widths[1],)),
        "a": jnp.zeros(n_in),
        "c": jnp.array(0.0),
    }

After initialization, define the ICNN forward pass. It maps a query point $q$ to a scalar approximation $\hat M_{1/\rho}f(q)$.

In [ ]:
# Define the ICNN forward pass.
@jax.jit
def forward(params, x):
    # First hidden layer.
    z1 = jax.nn.softplus(params["U1"] @ x + params["b1"])

    z2 = jax.nn.softplus(jax.nn.softplus(params["W2"]) @ z1 + params["U2"] @ x + params["b2"])

    # The scalar output approximates the Moreau envelope M_{1/rho} f(x).
    return jax.nn.softplus(params["v"]) @ z2 + params["a"] @ x + params["c"]

Vectorize the ICNN value and input-gradient computations.

In [ ]:
# Evaluate the scalar ICNN on a batch of query points.
predict = jax.vmap(forward, in_axes=(None, 0))

# MEL-ADMM uses the envelope gradient, so we train and monitor d/dx forward(params, x).
grad_x = jax.jit(jax.vmap(jax.grad(forward, argnums=1), in_axes=(None, 0)))

With value and gradient prediction available, we define the loss function as follows

In [ ]:
# Define the loss function for training the ICNN.
@jax.jit
def loss_fn(params, X, y, g):
    # Match the scalar Moreau-envelope value.
    value_mse = jnp.mean((predict(params, X) - y) ** 2)

    # Match the envelope gradient.
    grad_mse = jnp.mean(jnp.sum((grad_x(params, X) - g) ** 2, axis=1))

    return value_mse + grad_weight * grad_mse, (value_mse, grad_mse)

To optimize the loss efficiently, we create a mini-batch iterator that shuffles query points, envelope values, and gradient targets together.

In [ ]:
# Set up the training loop to optimize the ICNN parameters.
def batch_iterator(X, y, g, batch_size, shuffle_key):
    # Shuffle the samples at the start of each epoch.
    permutation = jax.random.permutation(shuffle_key, X.shape[0])

    # Yield aligned mini-batches of query points, envelope values, and gradient targets.
    for start in range(0, X.shape[0], batch_size):
        indices = permutation[start:start + batch_size]
        yield X[indices], y[indices], g[indices]

Before saving the trained model, define a small serialization helper so trained JAX arrays can be exported to JSON and loaded by Julia.

In [ ]:
# Train the ICNN parameters using Optax.
def to_serializable(value):
    if isinstance(value, (jax.Array, jnp.ndarray, np.ndarray)):
        return value.tolist()
    if isinstance(value, dict):
        return {key: to_serializable(val) for key, val in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_serializable(val) for val in value]
    return value

Next, define the ICNN export format.

In [ ]:
# Save the trained ICNN parameters.
def save_model(params, path, rho):
    path.parent.mkdir(parents=True, exist_ok=True)

    export_params = {
        "U": [params["U1"], params["U2"]],
        "W": [jnp.zeros_like(params["U1"]), jax.nn.softplus(params["W2"])],
        "b": [params["b1"], params["b2"]],
        "v": jax.nn.softplus(params["v"]),
        "a": params["a"],
        "c": params["c"],
        "rho": rho,
    }

    with open(path, "w") as f:
        json.dump(to_serializable(export_params), f)

With the model and loss defined, choose AdamW as the optimizer for ICNN training.


In [ ]:
# Choose the AdamW optimizer for training.
optimizer = optax.adamw(learning_rate=learning_rate, weight_decay=l2_reg)

Now, let's define one JIT-compiled ICNN training step: evaluate the loss, compute gradients, update optimizer state, and apply parameter updates.

In [ ]:
# Compute loss and gradients, update optimizer state, then apply the parameter update.
@jax.jit
def train_step(params, opt_state, Xb, yb, gb):
    (loss, metrics), grads = jax.value_and_grad(loss_fn, has_aux=True)(params, Xb, yb, gb)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    return optax.apply_updates(params, updates), opt_state, loss, metrics

After defining the training step, initialize the ICNN parameters and optimizer state. The printed shapes verify that the model dimensions match the loaded training data.

In [ ]:
# Initialize model parameters and optimizer state after all helper functions are defined.
key = jax.random.PRNGKey(seed)
params = init_icnn(key, n_in=X.shape[1], widths=widths)
opt_state = optimizer.init(params)

print("ICNN shapes: U1", params["U1"].shape, "W2", params["W2"].shape, "U2", params["U2"].shape)

After that, we run mini-batch ICNN training. The log reports value MSE and gradient MSE, then the trained model is saved for the Julia benchmark section.


In [ ]:
# Run mini-batch training.
for epoch in range(1, epochs + 1):
    # Use a fresh shuffle key for each epoch.
    batch_key, key = jax.random.split(key)

    # One optimizer step per mini-batch.
    for Xb, yb, gb in batch_iterator(X, y, g, batch_size, batch_key):
        params, opt_state, loss, (value_mse, grad_mse) = train_step(params, opt_state, Xb, yb, gb)

    # Print occasional full-dataset metrics so the tutorial output stays compact.
    if epoch == 1 or epoch % max(1, epochs // 5) == 0:
        _, (value_mse, grad_mse) = loss_fn(params, X, y, g)
        print(f"epoch {epoch:2d}: value MSE={value_mse:.3e}, grad MSE={grad_mse:.3e}")

# Save a demo model for the Julia MEL-ADMM benchmark section.
save_model(params, Path("model") / "neco_mpc-rho=1-demo.json", rho)
print("Saved model to model/neco_mpc-rho=1-demo.json !")

# Benchmark
**Julia kernel**

After training the Moreau-envelope model, we return to Julia to evaluate whether the learned update actually accelerates the MPC solve. The benchmark compares classical solvers with ADMM-based learned solvers:

- [IPOPT](https://coin-or.github.io/Ipopt/) and [MadNLP](https://madsuite.org/MadNLP.jl/stable/): interior-point NLP baselines;
- [ADMM](https://web.stanford.edu/~boyd/papers/pdf/admm_distr_stats.pdf): classical ADMM without learning;
- MEL-ADMM: ADMM with the learned Moreau-envelope gradient;
- sMEL-ADMM: a split version that separates dynamics and bound projections.

We compare solve time and relative optimality gap. We also plot the sMEL-ADMM trajectory and convergence traces.

*For the main benchmark, we do not rely on the small demo dataset/model above. Instead, the benchmark loads pre-generated data and a pre-trained ICNN model produced from a larger offline dataset. This gives a more reliable comparison of solve time and optimality gap, while the earlier demo cells remain fast enough for users to run and understand the workflow.*

We first load the Julia dependencies, including the MEL-ADMM and sMEL-ADMM implementations.


In [ ]:
# Import Julia packages
import Pkg
Pkg.activate("/content/L2O_tutorial/LEAF")
Pkg.instantiate()

cd("/content/L2O_tutorial/LEAF")
using JSON3, LinearAlgebra, NPZ, Printf, SparseArrays, StaticArrays, NNlib
using Plots, StatsPlots, Statistics
using Base.Threads

import MathOptInterface as MOI
import ParametricOptInterface as POI

In [ ]:
include.(["setup_em.jl", "preprocess.jl", "solver_em.jl", "admm_em.jl", "mel_admm_em.jl"])

Next, set the benchmark thread configuration.

In [ ]:
# Configure runtime thread settings for benchmark.
benchmark_num_threads = 8
BLAS.set_num_threads(benchmark_num_threads)

@printf("Julia threads: %d, BLAS threads: %d\n", benchmark_num_threads, BLAS.get_num_threads())

With these runtime settings fixed, we then select the initial SOC and forecast window used by all benchmark solvers.

In [ ]:
# Select the initial state and forecast window used by all benchmark solvers.
initial_state = mpc_data.x0
load = Vector{Float64}(mpc_data.load_forecast[1:mpc_data.N])
gen = Vector{Float64}(mpc_data.gen_forecast[1:mpc_data.N])

Now define benchmark labels, number of test samples, tolerance settings, and the MEL mini-batch size used in learned-gradient evaluation.

In [ ]:
# Configure benchmark labels, repetitions, tolerances, and MEL batch size.
benchmark_labels = ["IPOPT", "MadNLP", "ADMM", "MEL-ADMM", "sMEL-ADMM"]
n_samples = 10
tol = 1e-3
max_opt_gap = 0.01
s_mb = 24
gc_frequency = 10 # Run garbage collection every 10 iterations to manage memory during the benchmark.

Before running solvers, we define `collect_solver_times`, which repeatedly calls a solver and returns solver-reported runtimes. The first run is discarded as warm-up.

In [ ]:
# Measure solver runtime and discard the first warm-up sample.
function collect_solver_times(func_return_time, args, n_sample)
    time_arr = zeros(Float64, n_sample)
    objective_value = zero(Float64)

    for sample in 1:n_sample
        _, time_arr[sample], objective_value = func_return_time(args...; verbose = false)
        if sample % gc_frequency == 0
            GC.gc()
        end
    end

    return time_arr[2:end], objective_value
end

To compare objective quality, define the relative optimality gap using the reference objective $J^\star$ extracted from classical solvers:

$$
g_{\mathrm{opt}} = 100\frac{|J-J^\star|}{|J^\star|}.
$$


In [ ]:
# Report relative optimality gaps against the reference optimal objective Jopt.
function objective_gap(objective)
    return 100 * abs((Jopt - objective) / Jopt)
end

For trajectory visualization later, we define the plotting helper for grid import, battery power, controllable load, and battery SOC over the MPC horizon.

In [ ]:
# Plot the four state/control trajectories.
function timeseries_plot(solution, data)
    time_grid = collect(0:data.dT:data.dT * (data.N - 1))

    plt = plot(layout = (2, 2), size = (900, 600))
    labels = ["(a)", "(b)", "(c)", "(d)"]
    units = ["kW", "kW", "kW", "%"]

    for i in 1:4
        series = i == 4 ? solution[i, :] .* 100 : solution[i, :]
        ylims_ = i == 3 ? (4, 6) : nothing

        plot!(plt[i], time_grid, series, xlabel = "Time (h)", ylabel = "", xlims = (0, data.dT * data.N), xticks = 0:2:24, lw = 2.0, legend = false, grid = true, tickfont = font(12), guidefont = font(14))

        if ylims_ !== nothing
            ylims!(plt[i], ylims_)
        end

        xmin, xmax = extrema(time_grid)
        ymin, ymax = ylims_ === nothing ? extrema(series) : ylims_
        xpos = xmin + 0.02 * (xmax - xmin)
        ypos = (ymin + ymax) / 2
        annotate!(plt[i], (xpos, ypos, text(labels[i], 14, :left)))

        ypos_unit = ymax + 0.05 * (ymax - ymin)
        annotate!(plt[i], (xpos, ypos_unit, text(units[i], 12, :left)))
    end

    return plt
end


 We load the trained ICNN model from the previous section into Julia and reconstruct the learned Moreau-envelope model used by MEL-ADMM and sMEL-ADMM.

In [ ]:
# Load the trained ICNN parameters used by MEL-ADMM and sMEL-ADMM.
model_path = joinpath(pwd(), "model", "neco_mpc-rho=1.json")
rho, mp = load_model(model_path)

model = ICNN(mp.U[1], mp.b[1], [ICNN_Layer(mp.U[i], mp.W[i], mp.b[i]) for i in 2:length(mp.U)], mp.v, mp.a, mp.c)

println("Loaded trained ICNN from $(model_path)")

After loading the learned model, call IPOPT and MadNLP solvers and use them as baselines for the problem.

In [ ]:
# Build direct nonlinear-programming solvers for the benchmark baseline.
ipopt_solver = mpc_eco_solver("Ipopt", mpc_data, tol)
madnlp_solver = mpc_eco_solver("MadNLP", mpc_data, tol)

In [ ]:
ENV["GRB_LICENSE_FILE"] = "/content/L2O_tutorial/LEAF/gurobi.lic"

using Gurobi

We implement vanilla ADMM for the energy management example.

In [ ]:
# Build vanilla ADMM.
prime_sol = prime_sol_struct("Ipopt", mpc_data)
aux_sol = aux_solver_eco("Gurobi", mpc_data)
admm_solver = ADMM_eco_iter(mpc_data, prime_sol, aux_sol)

Then, we build MEL-ADMM and sMEL-ADMM. Both use the learned gradient $\nabla\hat M_{1/\rho}f$, but sMEL-ADMM additionally splits dynamics and bound constraints.


In [ ]:
# Build MEL-ADMM solvers.
mgrad = gradient_struct(model, s_mb, dim)
mel_solver = LME_ADMM(mpc_data, mgrad, aux_sol)

split_aux_sol = dynamics_projection(mpc_data)
smel_solver = LME_ADMM_split(mpc_data, mgrad, split_aux_sol)

Let's wrap ADMM, MEL-ADMM, and sMEL-ADMM so they share the same benchmark call signature.

In [ ]:
# Wrap ADMM solvers.
function admm_benchmark_solver(init, load, gen; verbose = false)
    return admm_solver(init, load, gen, ADMM_callback; verbose = verbose)
end

function mel_benchmark_solver(init, load, gen; verbose = false)
    return mel_solver(init, load, gen, ADMM_callback; verbose = verbose)
end

function smel_benchmark_solver(init, load, gen; verbose = false)
    return smel_solver(init, load, gen, sLME_ADMM_callback; verbose = verbose)
end

Now run repeated solves for IPOPT, MadNLP, ADMM, MEL-ADMM, and sMEL-ADMM on the same problem instance.

In [ ]:
# Run repeated solves and store each method's objective value from the final solve.
timing_ms = Dict{String, Vector{Float64}}()
objectives = Dict{String, Float64}()

ipopt_times, objectives["IPOPT"] = collect_solver_times(ipopt_solver, (initial_state, load, gen), n_samples)
madnlp_times, objectives["MadNLP"] = collect_solver_times(madnlp_solver, (initial_state, load, gen), n_samples)
admm_times, objectives["ADMM"] = collect_solver_times(admm_benchmark_solver, (initial_state, load, gen), n_samples)
mel_times, objectives["MEL-ADMM"] = collect_solver_times(mel_benchmark_solver, (initial_state, load, gen), n_samples)
smel_times, objectives["sMEL-ADMM"] = collect_solver_times(smel_benchmark_solver, (initial_state, load, gen), n_samples)

Convert run times from seconds to milliseconds for easier comparison.

In [ ]:
# Convert solver-reported times from seconds to milliseconds for display.
timing_ms["IPOPT"] = ipopt_times .* 1000
timing_ms["MadNLP"] = madnlp_times .* 1000
timing_ms["ADMM"] = admm_times .* 1000
timing_ms["MEL-ADMM"] = mel_times .* 1000
timing_ms["sMEL-ADMM"] = smel_times .* 1000

Now let's print each method's median solve time and relative optimality gap.


In [ ]:
# Print median solve time and relative optimality gap for each benchmark method.
for label in benchmark_labels
    @printf("%-10s median solve time = %7.3f ms, optimality gap = %7.4f%%\n",
            label, median(timing_ms[label]), objective_gap(objectives[label]))
end

We run sMEL-ADMM once more to store the full trajectory used for visualization.


In [ ]:
# Run sMEL-ADMM once to collect the trajectory used in the time-series plot.
split_solution, _, split_objective = smel_solver(initial_state, load, gen, nothing; verbose = false)
@printf("sMEL-ADMM objective = %.3f, optimality gap = %.4f%%\n", split_objective, objective_gap(split_objective))

Using the stored trajectory, plot the sMEL-ADMM control and state evolution over the MPC horizon.

In [ ]:
# Visualize the sMEL-ADMM state and control trajectory.
timeseries_plot(split_solution, mpc_data)

To study convergence behavior, set the iteration budgets used when plotting optimality-gap traces.

In [ ]:
# Use a fixed iteration budget for all optimality-gap traces.
Ipopt_n_iter = 40
LADMM_n_iter = 40
sLADMM_n_iter = 40

First, record IPOPT's optimality-gap trace using a callback.

In [ ]:
# Record IPOPT's relative optimality-gap trace through its callback.
ipopt_cb = callback_struct()
ipopt_trace_solver = mpc_eco_solver("Ipopt", mpc_data, 1e-6, ipopt_cb)
ipopt_trace_solver(initial_state, load, gen; verbose = false)

Next, record MEL-ADMM's optimality-gap trace using the ADMM callback.

In [ ]:
# Record MEL-ADMM's relative optimality-gap trace through the ADMM callback.
mel_cb = callback_struct()
mel_solver(initial_state, load, gen, (args...) -> ADMM_callback_iter(args..., mel_cb); verbose = false)

Finally, record sMEL-ADMM's optimality-gap trace using the split-ADMM callback.

In [ ]:
# Record sMEL-ADMM's relative optimality-gap trace through the split-ADMM callback.
smel_cb = callback_struct()
smel_solver(initial_state, load, gen, (args...) -> sADMM_callback_iter(args..., smel_cb); verbose = false)

Before plotting, clean callback traces so all methods use consistent iteration indices.

In [ ]:
# Remove callback entries before iteration one and optionally shift IPOPT to one-based iteration labels.
function iteration_trace(cbs; shift_zero = false)
    iter = shift_zero ? cbs.n_iter .+ 1 : cbs.n_iter
    keep = iter .>= 1
    return iter[keep], cbs.rel_opt_gap[keep]
end

Then convert callback data into arrays of iteration numbers and relative optimality gaps.

In [ ]:
# Convert callback state into plottable iteration and optimality-gap arrays.
ipopt_iter, ipopt_gap = iteration_trace(ipopt_cb; shift_zero = true)
mel_iter, mel_gap = iteration_trace(mel_cb)
smel_iter, smel_gap = iteration_trace(smel_cb)

Plot relative optimality gap versus iteration for IPOPT, MEL-ADMM, and sMEL-ADMM on a log scale.

In [ ]:
# Compare relative optimality-gap traces for IPOPT, MEL-ADMM, and sMEL-ADMM.
gap_plot = plot(xlabel = "iteration",
                ylabel = "optimality gap (%)",
                title = "Relative optimality gap",
                lw = 2,
                marker = :circle,
                yscale = :log10,
                size = (760, 420))

plot!(gap_plot, ipopt_iter, ipopt_gap, label = "IPOPT")
plot!(gap_plot, mel_iter, mel_gap, label = "MEL-ADMM")
plot!(gap_plot, smel_iter, smel_gap, label = "sMEL-ADMM")

gap_plot